# SDM Project — Neo4j Queries
This notebook contains the 4 analytical queries over the graph database.

In [1]:
from neo4j import GraphDatabase
import pandas as pd

NEO4J_URI      = "neo4j://127.0.0.1:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "sdmproject"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print("Connected to Neo4j")

Connected to Neo4j


---
## Query 1 — Top 3 most cited papers of each conference/workshop

In [3]:
query1 = """
MATCH (c:Conference)<-[:BELONGS_TO]-(e:Edition)<-[:PUBLISHED_IN]-(p:Paper)
WITH c, p, COUNT { ()-[:CITES]->(p) } AS times_cited
ORDER BY c.name, times_cited DESC
WITH c, collect({title: p.title, times_cited: times_cited})[0..3] AS top3
RETURN c.name AS conference, c.type AS type, top3
ORDER BY conference
"""

with driver.session() as session:
    results = session.run(query1).data()

rows = []
for r in results:
    for rank, paper in enumerate(r['top3'], start=1):
        rows.append({
            'conference':   r['conference'],
            'type':         r['type'],
            'rank':         rank,
            'title':        paper['title'],
            'times_cited':  paper['times_cited']
        })

df1 = pd.DataFrame(rows)
print(f"Total conferences/workshops: {df1['conference'].nunique()}")
df1.head(15)

Total conferences/workshops: 5


,conference,type,rank,title,times_cited
0,DAC,Conference,1,Combining Deterministic and Genetic Approaches...,5
1,DAC,Conference,2,On Test Set Preservation of Retimed Circuits.,5
2,DAC,Conference,3,CAD Methodology for the Design of UltraSPARC-I...,5
3,EGC (best of volume),Workshop,1,A New Way for Hierarchical and Topological Clu...,4
4,EGC (best of volume),Workshop,2,Ontology-Based Formal Specifications for User-...,4
5,EGC (best of volume),Workshop,3,A Bayesian Criterion for Evaluating the Robust...,3
6,PT-AI,Conference,1,Artificial Intelligence Safety Engineering: Wh...,5
7,PT-AI,Conference,2,"""Computational Ontology and Deontology"".",5
8,PT-AI,Conference,3,Introducing Experion as a Primal Cognitive Uni...,5
9,SLIP@DAC,Workshop,1,Exploiting PDN noise to thwart correlation pow...,4


---
## Query 2 — Community of each conference/workshop
Authors that have published papers on that conference/workshop in at least 4 different editions.

In [4]:
query2 = """
MATCH (a:Author)-[:WRITES]->(p:Paper)-[:PUBLISHED_IN]->(e:Edition)-[:BELONGS_TO]->(c:Conference)
WITH c, a, count(DISTINCT e) AS editions_count
WHERE editions_count >= 4
RETURN c.name       AS conference,
       c.type       AS type,
       a.name       AS author,
       editions_count
ORDER BY conference, editions_count DESC
"""

with driver.session() as session:
    results = session.run(query2).data()

df2 = pd.DataFrame(results)
print(f"Total community members found: {len(df2)}")
print(f"Conferences with community:    {df2['conference'].nunique()}")
df2.head(15)

Total community members found: 564
Conferences with community:    1


,conference,type,author,editions_count
0,DAC,Conference,Jason Cong,26
1,DAC,Conference,Kaushik Roy 0001,24
2,DAC,Conference,Alberto L. Sangiovanni-Vincentelli,24
3,DAC,Conference,Yao-Wen Chang,22
4,DAC,Conference,Andrew B. Kahng,21
5,DAC,Conference,Massoud Pedram,19
6,DAC,Conference,Anand Raghunathan,19
7,DAC,Conference,Miodrag Potkonjak,18
8,DAC,Conference,Kwang-Ting Cheng,18
9,DAC,Conference,Luca Benini,18


---
## Query 3 — Impact Factor of journals
Impact factor = citations received in a reference year (Y) by papers published in Y-1 and Y-2,
divided by the number of papers published in Y-1 and Y-2.

The `citation_year` property on the `CITES` edge is used to filter citations within the 2-year window.

In [5]:
query3 = """
// Step 1: find the latest year available per journal
MATCH (p:Paper)-[:PUBLISHED_IN]->(v:Volume)-[:BELONGS_TO]->(j:Journal)
WHERE p.year IS NOT NULL
WITH j, max(p.year) AS ref_year

// Step 2: count papers published in ref_year-1 and ref_year-2
MATCH (p2:Paper)-[:PUBLISHED_IN]->(v2:Volume)-[:BELONGS_TO]->(j)
WHERE p2.year = ref_year - 1 OR p2.year = ref_year - 2
WITH j, ref_year, count(p2) AS papers_published, collect(p2.id) AS paper_ids

// Step 3: count citations to those papers whose citation_year = ref_year
MATCH ()-[c:CITES]->(cited:Paper)
WHERE cited.id IN paper_ids
  AND c.citation_year = ref_year
WITH j.name          AS journal,
     ref_year,
     papers_published,
     count(c)        AS citations_in_window

RETURN journal,
       ref_year,
       papers_published,
       citations_in_window,
       round(toFloat(citations_in_window) / papers_published, 3) AS impact_factor
ORDER BY impact_factor DESC
"""

with driver.session() as session:
    results = session.run(query3).data()

df3 = pd.DataFrame(results)
print(f"Journals with impact factor: {len(df3)}")
df3

Journals with impact factor: 7


,journal,ref_year,papers_published,citations_in_window,impact_factor
0,SIGIR Forum,2025,30,417,13.900
1,IEEE Ann. Hist. Comput.,2025,33,172,5.212
2,Inf. Syst. Frontiers,2025,190,959,5.047
3,Int. J. Neural Syst.,2026,114,564,4.947
4,Int. J. Learn. Technol.,2025,22,103,4.682
5,Comput. Optim. Appl.,2026,77,77,1.000
6,IEEE Trans. Educ.,2025,62,52,0.839


---
## Query 4 — H-index of authors
H-index = the largest number h such that the author has at least h papers
each cited at least h times.

In [6]:
query4 = """
MATCH (a:Author)-[:WRITES]->(p:Paper)
WITH a, p, COUNT { ()-[:CITES]->(p) } AS citations
ORDER BY a.name, citations DESC
WITH a, collect(citations) AS citation_list
WITH a, citation_list, size(citation_list) AS total
WITH a, [i IN range(0, total - 1)
         WHERE citation_list[i] >= i + 1 | i + 1] AS h_values
RETURN a.name AS author,
       CASE WHEN size(h_values) > 0 THEN last(h_values) ELSE 0 END AS h_index
ORDER BY h_index DESC
LIMIT 50
"""

with driver.session() as session:
    results = session.run(query4).data()

df4 = pd.DataFrame(results)
print(f"Top 50 authors by h-index:")
df4

Top 50 authors by h-index:


,author,h_index
0,Ferrante Neri,10
1,Fangzhou Xu,10
2,Juan Manuel Górriz,9
3,Yogesh K. Dwivedi,9
4,Weidong Zhou,9
5,Nicola Ferro 0001,9
6,Hong Peng 0001,9
7,James W. Cortada,9
8,Adam Jatowt,9
9,Francesco Carlo Morabito,8


In [7]:
driver.close()
print("Connection closed")

Connection closed
